# Semana 10 — Da EC2 pública à arquitetura de rede

Partindo da implantação da Semana 07, vamos rastrear como uma requisição atravessa VPC, sub-rede, tabela de rotas, Internet Gateway, Security Group, sistema operacional e Docker.

## Resultados de aprendizagem

- localizar CIDR, sub-rede, rota, Internet Gateway e Security Group;
- explicar o que torna uma sub-rede pública;
- separar roteamento, IP público e autorização de porta;
- rastrear entrada e resposta entre navegador e contêiner;
- diagnosticar falhas por camada e comparar AWS, Azure, GCP e Oracle Cloud.

![Fluxo de uma VPC pública](images/semana_10_fluxo_vpc_publica.png)

A figura resume o caminho do tráfego entre a Internet, o Internet Gateway, a EC2 e os contêineres.

## 1. Arquitetura existente

A implantação usa uma região, uma Zona de Disponibilidade, uma VPC, uma sub-rede, IP privado e público, Security Group e Docker Compose. A aula não recria a EC2: ela lê e explica a arquitetura.

## 2. Camadas que não devem ser confundidas

| Camada | Pergunta | Exemplo |
|---|---|---|
| Endereçamento | Qual é a rede e o host? | `10.0.1.25/24` |
| Roteamento | Para onde enviar? | `0.0.0.0/0 → IGW` |
| Acesso | Origem pode usar a porta? | TCP 1880 |
| SO | Há processo escutando? | `ss -lntp` |
| Contêiner | Porta foi publicada? | `1880:1880` |
| Aplicação | A rota responde? | HTTP `/health` |

Abrir uma porta no Security Group não cria rota, não inicia processo e não publica porta Docker.

## 3. CIDR e sub-redes

Em `10.0.1.0/24`, os primeiros 24 bits identificam a rede. Uma VPC `10.0.0.0/16` contém várias sub-redes `/24`. A AWS reserva cinco endereços IPv4 por sub-rede.

In [ ]:
from ipaddress import ip_network, ip_address
vpc = ip_network('10.0.0.0/16')
subnet = ip_network('10.0.1.0/24')
instancia = ip_address('10.0.1.25')
print('sub-rede na VPC:', subnet.subnet_of(vpc))
print('instância na sub-rede:', instancia in subnet)
print('IPs totais:', subnet.num_addresses, '| estimativa AWS:', subnet.num_addresses - 5)

## 4. Tabela de rotas e maior prefixo

| Destino | Alvo | Significado |
|---|---|---|
| `10.0.0.0/16` | `local` | tráfego interno da VPC |
| `0.0.0.0/0` | Internet Gateway | demais destinos IPv4 |

O roteador escolhe a rota mais específica: `/16` vence `/0` para um destino interno.

In [ ]:
from ipaddress import ip_network, ip_address
rotas = [(ip_network('10.0.0.0/16'), 'local'), (ip_network('0.0.0.0/0'), 'internet-gateway')]
def escolher_rota(destino):
    ip = ip_address(destino)
    return max(((r.prefixlen, alvo, r) for r, alvo in rotas if ip in r), key=lambda x: x[0])
for destino in ['10.0.2.10', '8.8.8.8']:
    print(destino, '->', escolher_rota(destino))

## 5. O que torna uma sub-rede pública?

Uma sub-rede é pública quando sua tabela tem rota para um Internet Gateway. Para uma instância ser alcançável da Internet, ainda são necessários: IGW anexado, rota adequada, IPv4 público ou Elastic IP, regra de entrada no Security Group, serviço escutando e porta publicada pelo contêiner. NACLs customizadas também podem interferir.

## 6. Security Group

Security Groups são stateful: respostas de conexões permitidas são aceitas automaticamente. As regras devem seguir menor privilégio: SSH/22 restrito ao IP administrativo `/32`, HTTP/80 público quando necessário e portas de administração ou MQTT limitadas a origens confiáveis.

O Security Group autoriza tráfego; ele não substitui rota, processo no SO ou publicação Docker.

In [ ]:
from ipaddress import ip_network, ip_address
regras = [(22, ip_network('203.0.113.27/32'), 'SSH administrativo'), (80, ip_network('0.0.0.0/0'), 'HTTP público'), (1880, ip_network('198.51.100.0/24'), 'Node-RED institucional')]
def permitido(origem, porta):
    ip = ip_address(origem)
    return [nome for p, rede, nome in regras if p == porta and ip in rede]
for tentativa in [('203.0.113.27',22), ('198.51.100.50',1880), ('192.0.2.10',1880)]: print(tentativa, '->', permitido(*tentativa) or 'BLOQUEADO')

## 7. Rastreamento e diagnóstico

Para `http://IP_PUBLICO:1880`: navegador → IGW → rota → interface da EC2 → Security Group → TCP no SO → porta publicada pelo Docker → contêiner → resposta.

`timeout` sugere rota ou regra; `connection refused` indica host alcançado sem serviço; ausência de `0.0.0.0:1880->1880` no `docker ps` indica falta de binding; HTTP 404 significa que a rede funcionou, mas a rota da aplicação não existe.

## Atividade

Produza um inventário da implantação da Semana 07: CIDR da VPC e sub-rede, tabela de rotas, IPs, regras do Security Group, portas Docker e caminho completo até Node-RED ou Flask. Depois, formule três falhas em camadas diferentes e indique a evidência que as distingue.

## Equivalentes conceituais

| Conceito | AWS | Azure | GCP | Oracle |
|---|---|---|---|---|
| Rede virtual | VPC | VNet | VPC Network | VCN |
| Firewall | Security Group | NSG | VPC Firewall | Security List/NSG |
| Rotas | Route Table | Route Table/UDR | Routes | Route Table |

Os nomes variam; o raciocínio arquitetural é transferível.

## Síntese

VPC define o domínio; sub-redes segmentam endereços; rotas escolhem caminhos; Security Groups autorizam; o SO e o Docker precisam publicar o serviço. O diagnóstico eficiente segue o pacote, camada por camada.